In [ ]:
# TemporalBench v3 — Noise Injection Tier
# Difficulty: High-attention distractor facts, DecayTrap, OverlapTrap
# Task families: OverlapTrap, DecayTrap, PastQueryTrap, CurrentQuery, CausalQuery

import json
import numpy as np
from typing import Dict, List, Optional

In [ ]:
class Fact:
    def __init__(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        self.key = key
        self.value = value
        self.valid_from = valid_from
        self.valid_to = valid_to
        self.accesses = accesses

    def is_valid_at(self, day: int) -> bool:
        return self.valid_from <= day < self.valid_to

    def age(self, day: int) -> float:
        return day - self.valid_from

    def decay_score(self, day: int, half_life: float = 50.0) -> float:
        return np.exp(-0.693 * self.age(day) / half_life)

    def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)


class TemporalAttentionStore:
    """System D — decay + attention baseline. Fails on DecayTrap."""
    def __init__(self, half_life: float = 50.0, temporal_weight: float = 0.9, attention_weight: float = 0.1):
        self.half_life = half_life
        self.temporal_weight = temporal_weight
        self.attention_weight = attention_weight
        self.facts: Dict[str, List[Fact]] = {}

    def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))

    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        scores = [(self.temporal_weight * f.decay_score(as_of_day, self.half_life) +
                   self.attention_weight * f.attention_score(), f) for f in candidates]
        return max(scores, key=lambda x: x[0])[1].value


class TemporalAttentionStoreWithValidity:
    """System D_revised — hard validity gate. Unaffected by DecayTrap."""
    def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}

    def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))

    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        scores = [(f.decay_score(as_of_day, self.half_life) * f.attention_score(), f) for f in candidates]
        return max(scores, key=lambda x: x[0])[1].value

In [ ]:
def load_benchmark(path: str) -> tuple:
    with open(path + '_questions.jsonl') as f:
        questions = [json.loads(line) for line in f]
    with open(path + '_facts.jsonl') as f:
        facts_data = [json.loads(line) for line in f]
    with open(path + '_events.jsonl') as f:
        events_data = [json.loads(line) for line in f]
    return questions, facts_data, events_data


def extract_day(d: dict, default=None):
    """Try multiple field names for day/time values."""
    for field in ["day", "t_event", "as_of_day", "t", "time", "timestamp"]:
        val = d.get(field)
        if isinstance(val, (int, float)):
            return int(val)
    return default


def build_store(facts_data: List[Dict], events_data: List[Dict], store_class) -> tuple:
    store = store_class()
    for fact in facts_data:
        store.put(fact['key'], fact['value'], fact['valid_from'],
                  fact.get('valid_to') if fact.get('valid_to') is not None else 2**31 - 1,
                  fact.get('accesses', 0))
    event_log = sorted(events_data, key=lambda x: extract_day(x))
    return store, event_log


def replay_events(store, event_log: List[Dict], up_to_day: int):
    up_to_day = int(up_to_day)
    for ev in event_log:
        t_event = extract_day(ev)
        if t_event is None:
            continue
        if t_event > up_to_day:
            break
        ev_type = ev.get('type') or ev.get('event_type')
        if ev_type == 'FACT_OBSERVED' or ev_type == 'fact_access':
            subject = ev.get('subject')
            if not subject:
                continue
            domain = ev.get('domain', '')
            key = f"{domain}:{subject}" if domain else subject
            if key not in store.facts:
                continue
            val = ev.get('value')
            for f in store.facts[key]:
                if val is not None and f.value == val and f.valid_from <= t_event <= f.valid_to:
                    f.accesses = ev.get('accesses', f.accesses)

In [ ]:
def score_question(question: Dict, predicted: Optional[str]) -> float:
    answer = question.get('answer') or question.get('expected_answer') or ''
    if not answer:
        return 1.0 if predicted else 0.0
    return 1.0 if predicted == answer else 0.0


def run_evaluation(questions: List[Dict], facts_data: List[Dict], events_data: List[Dict],
                   store_class) -> Dict:
    store, event_log = build_store(facts_data, events_data, store_class)
    results = {}
    for q in questions:
        tf = q.get('task_family', 'Unknown')
        if tf not in results:
            results[tf] = {'correct': 0.0, 'total': 0}
        as_of_day = q.get('as_of_day') or q.get('day')
        key = f"{q.get('domain', '')}:{q.get('subject', '')}"
        replay_events(store, event_log, as_of_day)
        predicted = store.get(key, as_of_day)
        score = score_question(q, predicted)
        results[tf]['correct'] += score
        results[tf]['total'] += 1

    metrics = {tf: r['correct'] / r['total'] if r['total'] > 0 else 0.0 for tf, r in results.items()}
    trs = np.mean(list(metrics.values())) if metrics else 0.0
    return {'metrics': metrics, 'trs': trs}

In [ ]:
DATA_PATH = '/kaggle/input/datasets/zacharymaronek/temporalbenchmark/temporalbench/data/benchmarks/temporalbench_v3'

print('Loading TemporalBench v3 (noise injection tier)...')
questions, facts_data, events_data = load_benchmark(DATA_PATH)
print(f'Loaded {len(questions)} questions, {len(facts_data)} facts, {len(events_data)} events')

print('\nEvaluating D_revised (Validity Windows)...')
results_revised = run_evaluation(questions, facts_data, events_data, TemporalAttentionStoreWithValidity)
print(f'TRS: {results_revised["trs"]:.4f}')
for tf, acc in results_revised['metrics'].items():
    print(f'  {tf}: {acc:.1%}')

print('\nEvaluating D (Decay baseline)...')
results_d = run_evaluation(questions, facts_data, events_data, TemporalAttentionStore)
print(f'TRS: {results_d["trs"]:.4f}')
for tf, acc in results_d['metrics'].items():
    print(f'  {tf}: {acc:.1%}')

print('\n--- SUMMARY ---')
print(f'D_revised (Validity Windows): {results_revised["trs"]:.1%}')
print(f'D (Decay baseline):           {results_d["trs"]:.1%}')
print(f'Gap closed:                  {(results_revised["trs"] - results_d["trs"]):.1%}')